## Top/Bottom Shadowing Synthetic Data Exploration

### data can be generated with below commands

python -m simulator.simulate_data --n_samples 5000 --x_min -2278 --x_max 2278 --y_min -3456 --y_max -1878

python -m simulator.simulate_data --n_samples 5000 --x_min -2278 --x_max 2278 --y_min 1878 --y_max 3456

In [11]:
bot_single_row_syn_path = "/home2/erhan_bilgili/stix_flarelist_science/data/synthetic/sim_5000_x-2278-2278_y-3456--1878.npz"
top_single_row_syn_path = "/home2/erhan_bilgili/stix_flarelist_science/data/synthetic/sim_5000_x-2278-2278_y1878-3456.npz"

In [12]:
import numpy as np

bot_row_syn = np.load(bot_single_row_syn_path)
top_row_syn = np.load(top_single_row_syn_path)

In [13]:
bot_row_x = bot_row_syn['X']
bot_row_y = bot_row_syn['Y']

top_row_x = top_row_syn['X']
top_row_y = top_row_syn['Y']


## Single Row Top

For events that lie in the single row top regime, the bottom row should be illuminated more. The top/bot ratio is expected to be <1.0.

We look at the counts for detector 7c.

In [96]:
idx = 13
sample_top = top_row_x[idx]
sample_top_loc = top_row_y[idx]

print(f"Analyzing sample of location:\nX: {sample_top_loc[0]:.2f}\nY: {sample_top_loc[1]:.2f}")

Analyzing sample of location:
X: -57.75
Y: 3038.93


In [97]:
# get counts for detector 7c
pixels = sample_top[88:96]

In [98]:
# Syn det order -> Atop, abot, btop, bbot, ctop, cbot....

top_pixels = pixels[::2]
bot_pixels = pixels[1::2]

top_sum = np.sum(top_pixels)
bot_sum = np.sum(bot_pixels)
ratio = top_sum / bot_sum
print(f"top/bot ratio is: {ratio}")

top/bot ratio is: 0.25025906735751297


In [99]:
def get_ratios(n_events, syn_data, seed = 41):
    """print location and top/bot ratios for n random events"""
    np.random.seed(seed)
    X = syn_data['X']
    y = syn_data['Y']

    rnd_idx = np.random.randint(low=0, high=X.shape[0], size=n_events)
    
    for idx in rnd_idx:
        print(f"Event {idx}")
        sample = X[idx]
        print(f"X: {y[idx, 0]:.2f} Y: {y[idx, 1]:.2f}")

        pixels = sample[88:96]
        top_pixels = pixels[::2]
        bot_pixels = pixels[1::2]

        top_sum = np.sum(top_pixels)
        bot_sum = np.sum(bot_pixels)
        ratio = top_sum / bot_sum
        print(f"Top/Bot ratio is: {ratio:.2f}\n\n")

# N samples

In [100]:
n = 5
print(f"{n} random events, Single Row Top\n\n")
get_ratios(n, top_row_syn)

5 random events, Single Row Top


Event 1984
X: 103.49 Y: 2922.61
Top/Bot ratio is: 0.34


Event 931
X: 2036.56 Y: 2158.27
Top/Bot ratio is: 0.87


Event 3980
X: 1868.82 Y: 2242.42
Top/Bot ratio is: 0.72


Event 4066
X: -231.40 Y: 3138.50
Top/Bot ratio is: 0.23


Event 321
X: 1658.32 Y: 2455.38
Top/Bot ratio is: 0.69




# Single Row Bot

Expected Top Bot ratio for single row events are: ratio > 1.0

In [19]:
print(f"{n} random events, Single Row Bot\n\n")
get_ratios(n, bot_row_syn)

5 random events, Single Row Bot


Event 1984
X: -1433.52 Y: -3357.53
Top/Bot ratio is: 13.86


Event 931
X: 1375.42 Y: -2635.58
Top/Bot ratio is: 1.79


Event 3980
X: 1002.98 Y: -2292.10
Top/Bot ratio is: 1.04


Event 4066
X: 1094.64 Y: -2094.75
Top/Bot ratio is: 1.21


Event 321
X: 1734.30 Y: -3376.13
Top/Bot ratio is: 14.34




# Compare with real samples

In [20]:
import pandas as pd
df_raw = pd.read_csv("/home2/erhan_bilgili/stix_flarelist_science/data/6_final/stix_flarelist_final_20210214_20250830.csv")

In [21]:
from stix_train.config import REAL_DATA_PATH, get_feature_columns
real_df = pd.read_csv(REAL_DATA_PATH)
feat_cols = get_feature_columns(24)
TARGET_COLUMNS = ["loc_x_stix", "loc_y_stix"]

y_real = real_df[TARGET_COLUMNS].values.astype(float)
X_real = real_df[feat_cols].values.astype(float)


In [22]:
import numpy as np

y_loc = y_real[:, 1]
x_loc = y_real[:, 0]
full_x_cond = (x_loc < 2278) & (x_loc > -2278)
idxs_top = np.where(
    full_x_cond &
    (y_loc > 2300) & (y_loc < 3400)
)[0]

idxs_bot = np.where(
    full_x_cond & 
    (y_loc > -3400) & (y_loc < -2300)
)[0]

In [23]:
def get_ratios_real(n, real_x, real_y, seed=41):
    np.random.seed(seed)

    rnd_indx = np.random.randint(low=0, high=real_x.shape[0], size=n)

    for idx in rnd_indx:
        print(f"Event {idx}")
        sample = real_x[idx]
        print(f"X: {real_y[idx, 0]:.2f} Y: {real_y[idx, 1]:.2f}")

        pixels = sample[88:96]
        top_pixels = pixels[::2]
        bot_pixels = pixels[1::2]

        top_sum = np.sum(top_pixels)
        bot_sum = np.sum(bot_pixels)
        ratio = top_sum / bot_sum
        print(f"Top/Bot ratio is: {ratio:.2f}\n\n")
        


In [24]:
n = 5

# Real Top

In [25]:
get_ratios_real(n=n, real_x=X_real[idxs_top], real_y=y_real[idxs_top])

Event 80
X: 805.47 Y: 2592.08
Top/Bot ratio is: 0.55


Event 321
X: 785.90 Y: 3124.90
Top/Bot ratio is: 0.39


Event 243
X: -1072.22 Y: 3135.88
Top/Bot ratio is: 0.32


Event 601
X: -1668.44 Y: 2492.55
Top/Bot ratio is: 0.63


Event 407
X: -72.13 Y: 3093.75
Top/Bot ratio is: 1.01




# Real Bot

In [38]:
get_ratios_real(n=5, real_x=X_real[idxs_bot], real_y=y_real[idxs_bot], seed=1)

Event 37
X: -2083.34 Y: -2434.44
Top/Bot ratio is: 1.01


Event 235
X: -998.26 Y: -2979.19
Top/Bot ratio is: 2.25


Event 908
X: -13.54 Y: -2402.26
Top/Bot ratio is: 1.51


Event 72
X: 750.91 Y: -2455.86
Top/Bot ratio is: 1.58


Event 767
X: 96.42 Y: -3328.24
Top/Bot ratio is: 16.75




# Test implementation

In [69]:
yc= -3328.24

In [70]:
# placeholders
A,B,C,D = 100, 100, 100, 100


# instrument measurements
d_sep = 545.30
d_det = 47.7
D_tot = d_sep + d_det

# height and width of window (projection)
h_win = 20
w_win = 22


# height and width of detector
h_det = 9.2
w_det = 8.8



# 1. convert arcsec to radians
y_rad = np.deg2rad(yc/3600)

# 2. calculate how much the window has moved
dy = np.tan(y_rad) * D_tot


window_bot_border_y = dy - h_win/2
window_top_border_y = dy + h_win/2

top_edges_y = (h_det/2, window_top_border_y)
bot_edges_y = (-h_det/2, window_bot_border_y)



overlap_y = max(0, min(top_edges_y) - max(bot_edges_y) )

partial_overlap_y = overlap_y - h_det/2

# illumination ratio
fraction = partial_overlap_y / (h_det/2)



# -3477,   -1879
# we are in single row top regime, bot is more illuminated than top
if yc < 3477 and yc > 1879: 
    A_top, B_top, C_top, D_top = A * fraction, B * fraction, C * fraction, D * fraction
    A_bot, B_bot, C_bot, D_bot = A, B, C, D

# single row bot regime, top is more illuminated than bot
elif yc < -1879 and yc > -3477:
    A_bot, B_bot, C_bot, D_bot = A * fraction, B * fraction, C * fraction, D * fraction
    A_top, B_top, C_top, D_top = A, B, C, D

# else no shadowing
else:
    A_top, B_top, C_top, D_top, A_bot, B_bot, C_bot, D_bot = A, B, C, D, A, B, C , D


# fraction

In [71]:
float(fraction)

0.09362219771800186

In [72]:
1/float(fraction)

10.681227576093507

In [92]:
np.degrees(np.arctan2(h_win/2 - h_det/2, D_tot)) * 3600

np.float64(1878.2448021426585)

In [91]:
np.degrees(np.arctan2(h_win/2, D_tot)) * 3600

np.float64(3477.997595848036)